# SCAM DETECTOR
### AI-Powered SMS Spam and Suspicious Message Detection

**Hackathon Track:** AI + Finance

## Project Overview
Scam Detector is a hybrid message-analysis system designed to identify spam and suspicious SMS messages.

It combines machine learning with additional security checks to identify potentially harmful messages and provide safety guidance.

## How It Works
1. **Text Processing:** Converts SMS messages into numerical features using TF-IDF.
2. **Machine Learning:** Uses a Complement Naive Bayes classifier to identify spam-like patterns.
3. **Security Checks:** Detects warning signs such as password, PIN and OTP requests.
4. **Message Analysis:** Provides a warning and safety advice when suspicious patterns are detected.

## Dataset
UCI SMS Spam Collection — 5,572 original messages, with 5,169 unique messages remaining after duplicate removal.

## Model Performance
- Test accuracy: 96.52%
- Spam precision: 84%
- Spam recall: 89%
- Spam F1-score: 0.87

*Performance figures describe the machine-learning model on the held-out test set, not the additional security rules.*

## Important Limitation
The model was trained on a general English SMS dataset. It is an educational prototype and is not guaranteed to detect every scam, particularly region-specific financial fraud.

In [1]:

import urllib.request
import zipfile
import os

url = "https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip"

urllib.request.urlretrieve(url, "sms_dataset.zip")

with zipfile.ZipFile("sms_dataset.zip", "r") as zip_ref:
    zip_ref.extractall("sms_data")

print("Dataset downloaded successfully!")
print(os.listdir("sms_data"))


URLError: <urlopen error [Errno -3] Temporary failure in name resolution>

In [ ]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    for file in files:
        print(os.path.join(root, file))

In [ ]:

import pandas as pd

file_path = "/kaggle/input/datasets/veritasaihackathon/sms0spam/SMSSpamCollection"

df = pd.read_csv(
    file_path,
    sep="\t",
    header=None,
    names=["label", "message"]
)

print("Dataset loaded successfully!")
print("Total messages:", len(df))

df.head(10)


In [ ]:
print("Message counts:")
print(df["label"].value_counts())

print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicate messages:")
print(df.duplicated().sum())

In [ ]:
df = df.drop_duplicates().reset_index(drop=True)

print("Dataset after removing duplicates:")
print("Total messages:", len(df))
print("\nMessage counts:")
print(df["label"].value_counts())

In [ ]:
from sklearn.model_selection import train_test_split

X = df["message"]
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training messages:", len(X_train))
print("Testing messages:", len(X_test))

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB

model = Pipeline([
    ("tfidf", TfidfVectorizer()),
    ("classifier", MultinomialNB())
])

model.fit(X_train, y_train)

print("Scam Detector AI trained successfully!")

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

y_pred = model.predict(X_test)

print("MODEL ACCURACY:")
print(f"{accuracy_score(y_test, y_pred) * 100:.2f}%")

print("\nCLASSIFICATION REPORT:")
print(classification_report(y_test, y_pred))

print("\nCONFUSION MATRIX:")
print(confusion_matrix(y_test, y_pred))

In [ ]:
from sklearn.naive_bayes import ComplementNB
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

model2 = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1, 2))),
    ("classifier", ComplementNB())
])

model2.fit(X_train, y_train)

y_pred2 = model2.predict(X_test)

print("NEW MODEL ACCURACY:")
print(f"{accuracy_score(y_test, y_pred2) * 100:.2f}%")

print("\nCLASSIFICATION REPORT:")
print(classification_report(y_test, y_pred2))

print("\nCONFUSION MATRIX:")
print(confusion_matrix(y_test, y_pred2))

In [ ]:
def check_message(message):
    prediction = model2.predict([message])[0]

    if prediction == "spam":
        print("⚠️ WARNING: THIS MESSAGE LOOKS LIKE SPAM!")
        print("Be careful. Do not click suspicious links or share personal information.")
    else:
        print("✅ This message looks legitimate.")
        print("Still be cautious with unexpected requests.")

    print("\nMessage:", message)


check_message("Congratulations! You have won a free prize. Click here to claim now!")

In [ ]:
test_messages = [
    "Hey, are we still meeting tomorrow?",
    "URGENT! You have won a free iPhone. Claim your prize now!",
    "Can you please send me the homework notes?",
    "Your account is blocked. Click this link immediately to verify your details.",
    "Happy birthday! Hope you have an amazing day!"
]

for message in test_messages:
    print("\n" + "=" * 60)
    check_message(message)

In [ ]:
finance_tests = [
    "Your bank account has been suspended. Verify your identity immediately.",
    "Your transaction of Rs. 2500 was successful.",
    "Congratulations! You have won Rs. 50000. Send your bank details to claim.",
    "Your payment is due tomorrow. Please remember to pay your bill."
]

for message in finance_tests:
    print("\n" + "=" * 60)
    check_message(message)

In [ ]:
def check_message(message):
    prediction = model2.predict([message])[0]

    spam_index = list(model2.classes_).index("spam")
    spam_score = model2.predict_proba([message])[0][spam_index]

    print("\n========== SCAM DETECTOR ==========")

    if prediction == "spam":
        print("⚠️ WARNING: POTENTIAL SPAM DETECTED")
        print("This message contains patterns associated with spam.")
        print("Avoid clicking suspicious links or sharing personal information.")

    else:
        print("✅ NO STRONG SPAM PATTERNS DETECTED")
        print("However, this does NOT guarantee that the message is safe.")
        print("Be careful with unexpected requests for money or personal details.")

    print(f"\nSpam score: {spam_score * 100:.2f}%")
    print("===================================")


message = input("Enter a message to check: ")
check_message(message)

In [ ]:
test_message = "Your bank account will be closed today. Click here to confirm your password."

spam_index = list(model2.classes_).index("spam")
score = model2.predict_proba([test_message])[0][spam_index]

print("Spam score:", round(score * 100, 2), "%")
print("\nPredictions at different thresholds:")

for threshold in [0.50, 0.40, 0.30, 0.20, 0.15]:
    result = "POTENTIAL SPAM" if score >= threshold else "NO SPAM DETECTED"
    print(f"Threshold {threshold:.2f}: {result}")
    

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

spam_index = list(model2.classes_).index("spam")
spam_scores = model2.predict_proba(X_test)[:, spam_index]

print("THRESHOLD COMPARISON\n")

for threshold in [0.50, 0.40, 0.30, 0.25, 0.20, 0.15]:
    predictions = [
        "spam" if score >= threshold else "ham"
        for score in spam_scores
    ]

    precision = precision_score(y_test, predictions, pos_label="spam")
    recall = recall_score(y_test, predictions, pos_label="spam")
    f1 = f1_score(y_test, predictions, pos_label="spam")

    print(f"Threshold: {threshold:.2f}")
    print(f"Precision: {precision:.2f}")
    print(f"Recall:    {recall:.2f}")
    print(f"F1-score:  {f1:.2f}")
    print("-" * 30)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, precision_score, recall_score

# Create separate training and validation sets
X_subtrain, X_val, y_subtrain, y_val = train_test_split(
    X_train,
    y_train,
    test_size=0.2,
    random_state=42,
    stratify=y_train
)

# Train a temporary model
candidate_model = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1, 2))),
    ("classifier", ComplementNB())
])

candidate_model.fit(X_subtrain, y_subtrain)

# Get validation spam probabilities
spam_index = list(candidate_model.classes_).index("spam")
val_scores = candidate_model.predict_proba(X_val)[:, spam_index]

# Find the best threshold using validation data
best_threshold = 0
best_f1 = 0

for threshold in [0.50, 0.45, 0.40, 0.35, 0.30, 0.25]:
    predictions = [
        "spam" if score >= threshold else "ham"
        for score in val_scores
    ]

    f1 = f1_score(y_val, predictions, pos_label="spam")

    if f1 > best_f1:
        best_f1 = f1
        best_threshold = threshold

# Train final model on all training data
model_final = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1, 2))),
    ("classifier", ComplementNB())
])

model_final.fit(X_train, y_train)

print("Selected threshold:", best_threshold)
print("Validation F1-score:", round(best_f1, 3))
print("\nFinal model trained successfully!")

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

spam_index = list(model_final.classes_).index("spam")

test_scores = model_final.predict_proba(X_test)[:, spam_index]

final_predictions = [
    "spam" if score >= best_threshold else "ham"
    for score in test_scores
]

print("FINAL MODEL RESULTS")
print("===================")

print("Accuracy:", round(accuracy_score(y_test, final_predictions) * 100, 2), "%")

print("\nClassification Report:")
print(classification_report(y_test, final_predictions))

print("Confusion Matrix:")
print(confusion_matrix(y_test, final_predictions))

In [ ]:
def check_message(message):
    spam_index = list(model_final.classes_).index("spam")

    spam_score = model_final.predict_proba([message])[0][spam_index]

    print("\n========== SCAM DETECTOR ==========")

    if spam_score >= best_threshold:
        print("⚠️ POTENTIAL SPAM DETECTED")
        print("This message contains suspicious patterns.")
        print("Do not click unexpected links or share sensitive information.")
    else:
        print("✅ NO STRONG SPAM PATTERNS DETECTED")
        print("This does not guarantee that the message is safe.")
        print("Always verify unexpected requests independently.")

    print(f"\nSpam score: {spam_score * 100:.2f}%")
    print("===================================")


message = input("Enter a message to check: ")
check_message(message)

In [ ]:
import re

def check_message(message):
    spam_index = list(model_final.classes_).index("spam")
    spam_score = model_final.predict_proba([message])[0][spam_index]

    warning_patterns = [
        r"confirm your password",
        r"share your password",
        r"share your pin",
        r"send your otp",
        r"share your otp",
        r"bank details",
        r"account will be closed",
        r"account has been suspended",
        r"verify your identity immediately",
        r"click here.*(verify|confirm|claim)"
    ]

    warnings = [
        pattern for pattern in warning_patterns
        if re.search(pattern, message, re.IGNORECASE)
    ]

    print("\n========== SCAM DETECTOR ==========")
    print(f"AI spam score: {spam_score * 100:.2f}%")

    if spam_score >= best_threshold or warnings:
        print("\n⚠️ SUSPICIOUS MESSAGE DETECTED")

        if spam_score >= best_threshold:
            print("- AI detected spam-like patterns.")

        if warnings:
            print("- Additional security warning signs found.")

        print("\nDo not share passwords, PINs or OTPs.")
        print("Verify unexpected requests through official channels.")

    else:
        print("\n✅ NO STRONG WARNING SIGNS DETECTED")
        print("This does not guarantee that the message is safe.")

    print("===================================")


message = input("Enter a message to check: ")
check_message(message)

In [ ]:
demo_messages = [
    "Your transaction of Rs. 2500 was successful.",
    "Hey, are we still meeting tomorrow?",
    "Send your OTP to receive your refund.",
    "Your account has been suspended. Verify your identity immediately.",
    "Your payment is due tomorrow. Please pay your bill."
]

for msg in demo_messages:
    print("\nMessage:", msg)
    check_message(msg)

In [ ]:
import re

security_rules = {
    "Password request": r"confirm your password|share your password",
    "PIN request": r"share your pin|send your pin",
    "OTP request": r"send your otp|share your otp|enter your otp",
    "Bank details request": r"bank details",
    "Account threat": r"account will be closed|account has been suspended",
    "Urgent identity verification": r"verify your identity immediately",
    "Suspicious link request": r"click here.*(verify|confirm|claim)"
}

def check_message(message):
    spam_index = list(model_final.classes_).index("spam")
    spam_score = model_final.predict_proba([message])[0][spam_index]

    warnings = []

    for name, pattern in security_rules.items():
        if re.search(pattern, message, re.IGNORECASE):
            warnings.append(name)

    print("\n========== SCAM DETECTOR ==========")
    print(f"AI spam score: {spam_score * 100:.2f}%")

    if spam_score >= best_threshold or warnings:
        print("\n⚠️ SUSPICIOUS MESSAGE DETECTED")

        if spam_score >= best_threshold:
            print("- AI detected spam-like patterns.")

        if warnings:
            print("\nWarning signs:")
            for warning in warnings:
                print("-", warning)

        print("\nSafety advice:")
        print("Never share passwords, PINs or OTPs.")
        print("Verify unexpected requests independently.")

    else:
        print("\n✅ NO STRONG WARNING SIGNS DETECTED")
        print("This does not guarantee that the message is safe.")

    print("===================================")